# Impossible travel classifier (synthetic)

Binary label when distance exceeds `FRAUD.IMPOSSIBLE_TRAVEL_KM` (50 km) while elapsed time is at most `FRAUD.IMPOSSIBLE_TRAVEL_MINUTES` (30 min), matching Oasis fraud constants.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT / "scripts"))

import pandas as pd
from synthetic_data import generate_impossible_travel_rows
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

df = generate_impossible_travel_rows(10000)
df["impossible_travel"].value_counts()

In [ ]:
# Raw fixes + time only. Labels in CSV used true endpoint; lat2/lng2 are observed (noisy).
features = ["lat1", "lng1", "lat2", "lng2", "elapsed_minutes"]
X, y = df[features], df["impossible_travel"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", RandomForestClassifier(n_estimators=120, max_depth=12, random_state=42, class_weight="balanced")),
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)
print("holdout accuracy", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

In [ ]:
import joblib
ROOT.joinpath("artifacts").mkdir(parents=True, exist_ok=True)
ROOT.joinpath("data/synthetic").mkdir(parents=True, exist_ok=True)
joblib.dump(pipe, ROOT / "artifacts" / "impossible_travel.joblib")
df.to_csv(ROOT / "data" / "synthetic" / "impossible_travel.csv", index=False)